# 02 - Directional Changes Analysis

Deep dive into the DC framework:
1. DC detection at multiple thresholds
2. Scaling law verification
3. Intrinsic time computation
4. Cross-market DC comparison
5. DC events correlated with geopolitical events

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np

from src.data_collection.stock_fetcher import StockDataFetcher
from src.data_collection.conflict_tracker import ConflictEventTracker
from src.directional_changes.dc_algorithm import DirectionalChangeDetector, MultiThresholdDC
from src.directional_changes.dc_features import DCFeatureExtractor
from src.directional_changes.intrinsic_time import IntrinsicTimeAnalyzer, CrossMarketIntrinsicTime
from src.analysis.dc_event_correlation import DCEventCorrelator
from src.visualization.dc_plots import DCVisualizer

viz = DCVisualizer()

## 1. Load Data

In [ ]:
fetcher = StockDataFetcher()

sp500 = fetcher.fetch_symbol('^GSPC', '2015-01-01', '2025-12-31')
nifty = fetcher.fetch_symbol('^NSEI', '2015-01-01', '2025-12-31')
hsi = fetcher.fetch_symbol('^HSI', '2015-01-01', '2025-12-31')

events_tracker = ConflictEventTracker()
all_events = events_tracker.get_combined_geopolitical_events()
event_dates = all_events['date'].astype(str).tolist()

## 2. DC Detection on S&P 500

In [ ]:
threshold = 0.02  # 2% threshold
detector = DirectionalChangeDetector(threshold)
prices = sp500['close']

summaries = detector.detect(prices)
dc_df = detector.to_dataframe(summaries)

print(f'DC events detected (theta={threshold}): {len(summaries)}')
print(f'Upturns: {(dc_df["dc_direction"] == "UPTURN").sum()}')
print(f'Downturns: {(dc_df["dc_direction"] == "DOWNTURN").sum()}')
print(f'Avg magnitude: {dc_df["dc_magnitude"].abs().mean():.4f}')
print(f'Avg OS/DC ratio: {dc_df["overshoot_ratio"].mean():.4f}')

fig = viz.plot_dc_events_on_price(prices, dc_df, 'S&P 500 with DC Events', all_events)
fig.show()

## 3. Scaling Laws

In [ ]:
multi_dc = MultiThresholdDC()

print('=== S&P 500 Scaling Laws ===')
sp_scaling = multi_dc.scaling_law_data(sp500['close'])
fig = viz.plot_scaling_laws(sp_scaling)
fig.show()
sp_scaling

## 4. Intrinsic Time Analysis

In [ ]:
analyzer = IntrinsicTimeAnalyzer(0.02)

intrinsic_time = analyzer.compute_intrinsic_time(prices)
tcf = analyzer.time_compression_factor(prices, window_days=30)

fig = viz.plot_intrinsic_time(prices, intrinsic_time, tcf)
fig.show()

# Activity bursts
bursts = analyzer.detect_activity_bursts(prices)
print(f'\nActivity bursts detected: {len(bursts)}')
bursts

## 5. DC-Geopolitical Event Correlation

In [ ]:
correlator = DCEventCorrelator(0.02)

# Temporal coincidence test
coincidence = correlator.temporal_coincidence(prices, event_dates)
print('=== Temporal Coincidence Test ===')
for k, v in coincidence.items():
    print(f'  {k}: {v}')

# Magnitude comparison
print('\n=== Magnitude Comparison ===')
magnitude = correlator.dc_magnitude_around_events(prices, event_dates)
for k, v in magnitude.items():
    print(f'  {k}: {v}')

## 6. Cross-Market DC Comparison

In [ ]:
cross_it = CrossMarketIntrinsicTime(0.02)

market_prices = {}
if not sp500.empty: market_prices['US'] = sp500['close']
if not nifty.empty: market_prices['India'] = nifty['close']
if not hsi.empty: market_prices['China'] = hsi['close']

if len(market_prices) >= 2:
    tcf_df = cross_it.cross_market_tcf(market_prices)
    print('Cross-market TCF correlation:')
    print(tcf_df.corr().round(4))
    
    # Contagion lag
    if 'US' in market_prices and 'India' in market_prices:
        lag = cross_it.contagion_lag(market_prices['US'], market_prices['India'])
        print(f'\nUS -> India contagion lag: {lag["optimal_lag_days"]} days (corr={lag["max_correlation"]:.4f})')